# Approximate Bayesian Computation (ABC)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import corner

%matplotlib inline
np.random.seed(384)

# Example 0
Let's return to the example from a while ago

> We point our telescope to observe the light from a single star with true flux $F_{\rm true}$, which remains constant with time. We will ignore any systematic errors. We perform $N$ measurements with our telescope, where the $i^{\rm th}$ measurement report an observed flux $F_i$ with uncertainty/error $e_i$.

In [ ]:
# start by generating a synthetic observation
# Here we take a single measurement of photon counts, F_obs ~ Poisson(F_true)
F_true = 1000
F_obs = np.random.poisson(lam=F_true)

We can also write down the simulation of the **data generation process**. In this case this is trivial because we know exact data generation process.

In [ ]:
def simulation(F_true): 
    return np.random.poisson(lam=F_true)

ABC first involves repeating two steps
1. sample the prior: $\theta'\sim p(\theta)$
2. run simulation: $\mathcal{D}' \sim F(\theta')$


For our prior, lets use
$$p(\theta) = \mathcal{N}(\mu=1002, \sigma=4)$$

In [ ]:
def sample_prior(): 
    # sample prior
    return 1002. + 4. * np.random.normal()

Let's repeat the process `Nsample = 100000` times 

In [ ]:
Nsample = 100000

samples = np.zeros((Nsample, 2))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.array([theta_p, data_p])

Now we have a bunch of $\{(\theta', \mathcal{D}')\}$ samples drawn from the joint distribution $p(\theta, \mathcal{D})$. 

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(samples[:,0], samples[:,1], s=1)
plt.xlabel(r'$\theta$', fontsize=25)
plt.ylabel(r'$\mathcal{D}$', fontsize=25)
plt.show()

Now to obtain the **posterior** from these samples, ABC approximates: 
$$p(\theta\,|\,\mathcal{D}_{\rm obs}) \approx p(\theta\,|\,\mathcal{D} \approx \mathcal{D}_{\rm obs})$$

We'll define

In [ ]:
close_enough = np.abs(samples[:,1] - F_obs) < 1
print('%i simulations have D approx D_obs' % np.sum(close_enough))

We can visualize the $\mathcal{D} \approx \mathcal{D}_{\rm obs}$ in the $p(\theta, \mathcal{D})$ distribution

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(samples[:,0], samples[:,1], s=1)
plt.scatter(samples[:,0][close_enough], samples[:,1][close_enough], s=1, c='C1')
plt.xlabel(r'$\theta$', fontsize=25)
plt.ylabel(r'$\mathcal{D}$', fontsize=25)
plt.show()

More importantly, we can examine their $\theta$ distribution to estimate the posterior

In [ ]:
plt.figure(figsize=(4,2))
plt.hist(samples[:,0][close_enough], range=(960, 1040), bins=40)
plt.axvline(F_true, color='k', linestyle='--')
plt.show()

# Example 1
Now that was a very simple 1D example. Lets consider a slightly more interesting example: our favorite, fitting a line to some data.

In [ ]:
a_true, b_true = 0, 1.75
x = np.array([1,1.5,2,2.5])
y = np.array([1.87640523, 2.59001572, 3.4978738 , 4.47408932])
yerr = 0.1

plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')
plt.show()

Same as before, lets set up the **simulation approximating the data generation process** and choose our **prior**

In [ ]:
def simulation(theta): 
    a, b = theta
    noise = yerr * np.random.normal(size=len(x))
    return a + b * x + noise

def sample_prior(): 
    # sample a and b from uniform prior
    _a = -0.5 + np.random.uniform() 
    _b = 10*np.random.uniform()
    return [_a, _b]

Let's generate a bunch of samples from $p(\theta, \mathcal{D})$

In [ ]:
Nsample = 500000

samples = np.zeros((Nsample, 6))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.concatenate([theta_p, data_p])

Now this is harder to visualize since $\theta$ is 2D and $\mathcal{D}$ is 4D --- $p(\theta, \mathcal{D})$ is a 6D distribution.

Regardless of dimensions, the rest of the procedure for ABC remains the same: 
1. keep only samples with $\mathcal{D} \approx \mathcal{D}_{\rm obs}$
2. examine the distribution of $\theta$

Since $\mathcal{D}$ is 4D, we will use euclidean distance $\rho(\mathcal{D}, \mathcal{D}') < \epsilon$ for (1). 

In [ ]:
rho = np.sqrt(np.sum((samples[:,2:] - y[None, :])**2, axis=1))
epsilon = 0.5
print('reject %f percent of simulations' % (100.*(1-np.mean(rho < epsilon))))

Now lets look at the $\theta$ distribution of the accepted samples --- i.e. the posterior estimate.

In [ ]:
fig = plt.figure(figsize=(3,3))
_ = corner.corner(samples[:,:2][rho < epsilon], labels=['a', 'b'], fig=fig)
corner.overplot_lines(fig, [a_true, b_true], color="C1")
corner.overplot_points(fig, np.array([a_true, b_true])[None], marker="s", color="C1")
plt.show()

You may have noticed that the posterior estimate is significantly broader than the posterior from MCMC from earlier. That's the "approximate" part of ABC and a consequence of relaxing $\mathcal{D} = \mathcal{D}'$ to $\rho(\mathcal{D}, \mathcal{D}') < \epsilon$. 

Now if we reduce $\epsilon$, we'll get closer to the true posterior

In [ ]:
epsilon = 0.2
print('reject %f percent of simulations' % (100.*(1-np.mean(rho < epsilon))))

In [ ]:
fig = plt.figure(figsize=(3,3))
_ = corner.corner(samples[:,:2][rho < epsilon], labels=['a', 'b'], fig=fig)
corner.overplot_lines(fig, [a_true, b_true], color="C1")
corner.overplot_points(fig, np.array([a_true, b_true])[None], marker="s", color="C1")
plt.show()

# in-class exericse: slope of the IMF
In Lecture 6 we drew stellar masses from the initial mass function by rejection sampling. 
Now let's go the other way: given measured masses, infer the slope. 
Take a power law between $m_{\min} = 1\,M_\odot$ and $m_{\max} = 20\,M_\odot$,

$$p(m\,|\,\alpha) = C(\alpha)\, m^{-\alpha}, \qquad C(\alpha) = \frac{1-\alpha}{m_{\max}^{1-\alpha} - m_{\min}^{1-\alpha}},$$

and suppose we have measured masses for $N = 100$ stars in a young cluster ($\alpha = 2.35$).
I've implemented the simulator below that draws `N_STAR = 100` masses for a given `alpha`. 
The observed data here is the set of 100 masses in a cluster. 

We'll run the simator to first generate our mock observations

In [ ]:
M_MIN, M_MAX, N_STAR = 1.0, 20.0, 100

def simulate_masses(alpha, n=N_STAR):
    """One simulated cluster of n stellar masses per value of alpha (rows)."""
    alpha = np.atleast_1d(alpha)[:, None]
    e = 1 - alpha
    u = np.random.uniform(size=(alpha.shape[0], n))
    return (M_MIN**e + u * (M_MAX**e - M_MIN**e))**(1 / e)

alpha_true = 2.35
m_obs = simulate_masses(alpha_true)[0]

fig, ax = plt.subplots(figsize=(5, 3))
bins = np.geomspace(M_MIN, M_MAX, 15)
ax.hist(m_obs, bins=bins, histtype='step', color='k', lw=1.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'mass [$M_\odot$]')
ax.set_ylabel('number of stars')
plt.show()

### exact posterior to check ABC

Here the likelihood can be derived analytically so we can compare ABC with the exact answer. 
With a flat prior $\alpha\sim\mathcal{U}(1.5, 3.5)$,

$$\log p(\{m_i\}\,|\,\alpha) = N\log C(\alpha) - \alpha\sum_{i=1}^N \log m_i .$$

In [ ]:
A_LO, A_HI = 1.5, 3.5
a_grid = np.linspace(A_LO, A_HI, 2001)

def log_like(alpha, m):
    e = 1 - alpha
    return len(m) * np.log(e / (M_MAX**e - M_MIN**e)) - alpha * np.log(m).sum()

lp = np.array([log_like(a, m_obs) for a in a_grid])
post_exact = np.exp(lp - lp.max())
post_exact /= np.trapezoid(post_exact, a_grid)
mu_exact = np.trapezoid(a_grid * post_exact, a_grid)
sd_exact = np.sqrt(np.trapezoid((a_grid - mu_exact)**2 * post_exact, a_grid))
print(f'exact posterior: alpha = {mu_exact:.3f} +/- {sd_exact:.3f}')

### 1. sample the $p(\theta, \mathcal{D})$ distribution
ABC requires sampling the joint parameter, observable distribution. We can do this as we did in the examples above:

1. sample the prior: $\theta'\sim p(\theta)$
2. run simulation: $\mathcal{D}' \sim F(\theta')$

Draw `n_sim = 200000` samples from $p(\theta, \mathcal{D})$. Use the same flat prior $p(\alpha) = \mathcal{U}(1.5, 3.5)$ as above.

In [ ]:
n_sim = 200000



### 2. choosing $\rho$
The full observable/observation is $\mathcal{D}_{\rm obs} = \{m_{1}, ..., m_{100}\}$, 100 dimensional vector. 
Since imposing the condition $\mathcal{D}_{\rm obs} \approx \mathcal{D}$ is difficult in 100 dimensional-space, we need to specify a distance metric $\rho(\mathcal{D}_{\rm obs}, \mathcal{D})$. 
Implement ABC using the following distance metrics: 

$$\mathcal{D}_{\rm obs} = \{\log m_{1, {\rm obs}}, ..., \log m_{100, {\rm obs}}\}$$
$$\mathcal{D}' = \{\log m'_{1}, ..., \log m'_{100}\}$$

1. the absolute difference of the most massive star --- $\rho = |\max(\mathcal{D}_{\rm obs}) - \max(\mathcal{D}')|$
2. the absolute difference of the mean log mass --- $\rho = |{\rm mean}(\mathcal{D}_{\rm obs}) - {\rm mean}(\mathcal{D}')|$
3. choose a distance metric
4. RMS difference of the sorted log masses$^1$ --- $\rho = \sqrt{{\rm mean}\big[\big({\rm sort}(\mathcal{D}_{\rm obs}) - {\rm sort}(\mathcal{D}')\big)^2\big]}$

For each distnace, set $\epsilon$ so that there's roughly ~1000 accepted samples. 
Plot the resulting posterior on $\alpha$. Plot the true posterior above for comparison. 

$^1$: sorting doesn't matter because of ordering of the stars carry no information 

### 3. $\epsilon$ threshold
Keep the mean log mass as the distance metric but vary $\epsilon$ so that acceptance fraction from 50% down to 0.02%. 
Plot how the median and standard deviation of the $\alpha$ posterior as a function of $\epsilon$. 
Compare it against the median and standard deviation of the true posterior.

### (Optional): More efficient ABC
ABC with rejection sampling is very inefficient. There are newer more efficient implementations of ABC. 
Below is pseudocode for ABC with Population Monte Carlo. Try implementing it and applying it to the IMF problem with the mean log mass distance metric.

```
# --- t = 0: plain rejection ABC with a loose threshold ---
choose N (e.g. 1000), T (e.g. 6), and a loose epsilon_0
for i in 1 ... N:
    rho = infinity
    while rho > epsilon_0:
        draw theta* ~ p(theta)                       # sample the prior
        simulate D* = F(theta*)
        rho = rho(D*, D_obs)
    theta^i_0 = theta*
    rho^i_0   = rho
    w^i_0     = 1/N

# --- t = 1 ... T: shrink epsilon, propose from the previous population ---
for t = 1 ... T:
    epsilon_t = median(rho^1_{t-1}, ..., rho^N_{t-1})     # shrink the threshold
    Sigma_t   = 2 * weighted variance of {theta_{t-1}} with weights {w_{t-1}}

    for i in 1 ... N:
        rho = infinity
        while rho > epsilon_t:
            draw theta** from {theta^j_{t-1}} with probability {w^j_{t-1}}
            perturb theta* ~ N(theta**, Sigma_t)          # kernel K
            if p(theta*) == 0: continue                   # outside the prior: reject, draw again
            simulate D* = F(theta*)
            rho = rho(D*, D_obs)
        theta^i_t = theta*
        rho^i_t   = rho

        # importance weight: prior / proposal density
        w^i_t = p(theta^i_t) / Sum_j [ w^j_{t-1} * K(theta^i_t | theta^j_{t-1}, Sigma_t) ]

    normalize: w^i_t <- w^i_t / Sum_k w^k_t

# posterior estimate: the weighted samples {theta^i_T, w^i_T}
```